# Notebook 1 — Exploração: Transfer Learning Eficiente

## Objetivos

1. Carregar e explorar o dataset de classificação de domínio textual
2. Analisar distribuições de texto e classes
3. Compreender a arquitetura de modelos pré-treinados (DistilBERT)
4. Visualizar a redução de parâmetros com LoRA (Hu et al., 2022)
5. Comparar técnicas PEFT teóricas (Tabela 2, material da Aula 5)

## Conexão Teórica

> *"O conceito de transfer learning fundamenta-se na premissa de que o conhecimento adquirido
> ao resolver uma tarefa pode ser transferido para facilitar a solução de tarefas relacionadas."*
> — Material da Aula 5, seção 'Fundamentos do Transfer Learning' (Pan & Yang, 2010)

> *"A dimensão intrínseca d de uma tarefa de fine-tuning pode ser formalmente definida como
> a menor dimensão de um subespaço no qual a otimização ainda converge para uma solução de
> qualidade comparável ao espaço completo."*
> — Material da Aula 5, seção 'Hipótese de Subespaços de Baixa Dimensionalidade'
> (Aghajanyan et al., 2021)

In [ ]:
# Imports
import sys
import os

# Adiciona raiz do projeto ao path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")

print("Imports carregados com sucesso.")

## 1. Geração e Carregamento do Dataset

Geramos um dataset sintético de **classificação de domínio textual** com 5 categorias:
jurídico, tecnologia, financeiro, saúde e educação.

Este cenário simula a adaptação de modelos pré-treinados a domínios especializados —
exatamente o caso de uso do Transfer Learning (Pan & Yang, 2010).

In [ ]:
from src.data_preprocessing import DataPreprocessor, generate_synthetic_dataset

# Gerar dataset (5000 amostras, 5 domínios)
DATASET_PATH = os.path.join("..", "data", "raw", "dataset.csv")
os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

df = generate_synthetic_dataset(n_samples=5000, random_state=42, output_path=DATASET_PATH)
print(f"Dataset gerado: {df.shape[0]} amostras, {df.shape[1]} colunas")
df.head(10)

## 2. Análise Exploratória do Dataset

Verificamos o balanceamento de classes e as estatísticas textuais.
Um dataset balanceado é importante para evitar viés ao treinar
adaptadores com poucos parâmetros (LoRA treina ~0.06% dos pesos).

In [ ]:
# Estatísticas básicas do dataset
print("=" * 50)
print("Estatísticas do Dataset")
print("=" * 50)
print(f"Total de amostras: {len(df)}")
print(f"Colunas: {list(df.columns)}")
print(f"\nDistribuição de classes:")
print(df["label"].value_counts())
print(f"\nTipos de dados:")
print(df.dtypes)

In [ ]:
# Distribuição de classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
class_counts = df["label"].value_counts().sort_index()
colors = ["#3498db", "#e74c3c", "#2ecc71", "#9b59b6", "#f39c12"]
class_counts.plot(kind="bar", ax=axes[0], color=colors)
axes[0].set_title("Distribuição de Classes", fontsize=13)
axes[0].set_xlabel("Domínio")
axes[0].set_ylabel("Contagem")
axes[0].tick_params(axis="x", rotation=45)

# Pizza
class_counts.plot(kind="pie", ax=axes[1], colors=colors, autopct="%1.1f%%")
axes[1].set_title("Proporção de Classes", fontsize=13)
axes[1].set_ylabel("")

plt.suptitle("Balanceamento do Dataset de Classificação de Domínio", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Análise de Comprimento dos Textos

O comprimento dos textos impacta diretamente a escolha do `max_length`
na tokenização. Modelos Transformer têm limite de sequência (512 para BERT),
e padding/truncamento afetam eficiência computacional.

In [ ]:
# Análise de comprimento textual
df["text_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de caracteres
for label in sorted(df["label"].unique()):
    subset = df[df["label"] == label]
    axes[0].hist(subset["text_length"], bins=30, alpha=0.6, label=label)
axes[0].set_title("Distribuição de Comprimento (caracteres)", fontsize=12)
axes[0].set_xlabel("Número de caracteres")
axes[0].set_ylabel("Frequência")
axes[0].legend()

# Boxplot de palavras por domínio
sns.boxplot(data=df, x="label", y="word_count", ax=axes[1], palette=colors)
axes[1].set_title("Palavras por Domínio", fontsize=12)
axes[1].set_xlabel("Domínio")
axes[1].set_ylabel("Número de palavras")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Comprimento médio: {df['text_length'].mean():.0f} chars, {df['word_count'].mean():.0f} palavras")
print(f"Comprimento máximo: {df['text_length'].max()} chars")

## 4. Pré-processamento com DataPreprocessor

Utilizamos a classe `DataPreprocessor` do módulo `src/data_preprocessing.py`
para preparar os dados: limpeza, codificação de labels e split estratificado.

In [ ]:
# Pipeline de pré-processamento
preprocessor = DataPreprocessor(max_length=128, test_size=0.2, val_size=0.1, random_state=42)

df_loaded = preprocessor.load_data(DATASET_PATH)
df_clean = preprocessor.clean_data()
df_features = preprocessor.prepare_features()
train_df, val_df, test_df = preprocessor.split_data()

print(f"Label map: {preprocessor.label_map}")
print(f"Número de classes: {preprocessor.num_labels}")
print(f"\nSplit dos dados:")
print(f"  Train: {len(train_df)} amostras")
print(f"  Val:   {len(val_df)} amostras")
print(f"  Test:  {len(test_df)} amostras")

## 5. Modelo Pré-treinado: Arquitetura e Parâmetros

Carregamos o **DistilBERT** como modelo base para explorar:
- Número total de parâmetros
- Arquitetura do transformer
- Camadas candidatas para LoRA

> *"As primeiras camadas de redes neurais aprendem features genéricas,
> enquanto camadas mais profundas especializam-se em features específicas
> do domínio."* — Material da Aula 5 (Yosinski et al., 2014)

In [ ]:
from src.model import PEFTModelWrapper
from src.utils import count_parameters, get_model_size_mb

# Carregar modelo base (DistilBERT para classificação)
wrapper = PEFTModelWrapper(
    model_name="distilbert-base-uncased",
    num_labels=preprocessor.num_labels,
    device="cpu",  # CPU para exploração
)
base_model = wrapper.load_base_model(quantize_4bit=False)

# Contagem de parâmetros do modelo base
base_params = count_parameters(base_model)
print(f"Modelo: distilbert-base-uncased")
print(f"Parâmetros totais: {base_params['total']:,}")
print(f"Tamanho estimado: {get_model_size_mb(base_model):.1f} MB")

In [ ]:
# Visualizar camadas do modelo (candidatas a LoRA)
print("Camadas do modelo (primeiros 30 módulos):")
print("=" * 60)
for i, (name, param) in enumerate(base_model.named_parameters()):
    if i >= 30:
        print("...")
        break
    trainable = "✓" if param.requires_grad else "✗"
    print(f"[{trainable}] {name:50s} {str(list(param.shape)):>20s}")

## 6. Aplicação de LoRA: Redução de Parâmetros Treináveis

Agora aplicamos **LoRA (Low-Rank Adaptation)** ao modelo. Conforme
Hu et al. (2022), LoRA injeta matrizes de baixo rank B e A nos
módulos de atenção, parametrizado como:

$$h = W_0 x + \Delta W x = W_0 x + BA x$$

onde $B \in \mathbb{R}^{d \times r}$ e $A \in \mathbb{R}^{r \times k}$
com rank $r \ll \min(d, k)$.

Para DistilBERT, aplicamos LoRA em `q_lin` e `v_lin` (projeções
de query e value no mecanismo de atenção).

In [ ]:
# Aplicar LoRA (r=16, alpha=32) — Snippet 2 do material
peft_model = wrapper.apply_lora(
    r=16,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
)

# Comparar parâmetros
peft_params = count_parameters(peft_model)
print(f"\n{'='*50}")
print(f"COMPARAÇÃO DE PARÂMETROS")
print(f"{'='*50}")
print(f"{'Métrica':<30} {'Base':>15} {'LoRA':>15}")
print(f"{'-'*60}")
print(f"{'Total':.<30} {base_params['total']:>15,} {peft_params['total']:>15,}")
print(f"{'Treináveis':.<30} {base_params['total']:>15,} {peft_params['trainable']:>15,}")
print(f"{'Congelados':.<30} {0:>15,} {peft_params['frozen']:>15,}")
print(f"{'% Treináveis':.<30} {'100.0000%':>15} {peft_params['trainable_pct']:>14.4f}%")

In [ ]:
# Visualização da redução de parâmetros
from src.training import compare_training_efficiency
from src.evaluation import plot_parameter_comparison

# Nota: para comparação, consideramos que no full FT todos parâmetros são treináveis
comparison_df = compare_training_efficiency(base_model, peft_model)
print("\nTabela Comparativa:")
print(comparison_df.to_string(index=False))

fig = plot_parameter_comparison(
    comparison_df,
    title="Comparação: Full Fine-Tuning vs LoRA (r=16)\n(Reproduz Tabela 2 do material — Hu et al., 2022)",
)
plt.show()

## 7. Exploração de Diferentes Ranks LoRA

O rank `r` controla o trade-off entre capacidade expressiva e eficiência.
Valores típicos variam de 4 a 64 (Hu et al., 2022, Seção 4).

Vamos visualizar como o número de parâmetros treináveis varia com `r`.

In [ ]:
# Explorar diferentes ranks
from peft import LoraConfig as PeftLoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

ranks = [2, 4, 8, 16, 32, 64]
results = []

for r in ranks:
    # Recarregar modelo base limpo
    temp_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=5
    )
    config = PeftLoraConfig(
        r=r, lora_alpha=2*r, target_modules=["q_lin", "v_lin"],
        lora_dropout=0.05, task_type=TaskType.SEQ_CLS,
    )
    temp_peft = get_peft_model(temp_model, config)
    stats = count_parameters(temp_peft)
    results.append({"rank": r, **stats})
    del temp_model, temp_peft

rank_df = pd.DataFrame(results)
print(rank_df[["rank", "trainable", "trainable_pct"]].to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(r) for r in ranks], rank_df["trainable"], color="#2ecc71")
ax.set_xlabel("Rank (r)", fontsize=12)
ax.set_ylabel("Parâmetros Treináveis", fontsize=12)
ax.set_title("Parâmetros Treináveis vs Rank LoRA\n(Hu et al., 2022 — r controla dimensão intrínseca)", fontsize=13)
for i, v in enumerate(rank_df["trainable"]):
    ax.text(i, v + 1000, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 8. Tabela Comparativa de Técnicas PEFT

Reproduzimos a **Tabela 2 do material da Aula 5**: *Comparação de
Técnicas PEFT para LLaMA-7B* (Hu et al., 2022; Dettmers et al., 2023).

Esta tabela mostra que LoRA e QLoRA oferecem o melhor trade-off entre
eficiência e desempenho, sem latência adicional após fusão de pesos.

In [ ]:
from src.evaluation import create_peft_comparison_table

peft_table = create_peft_comparison_table()
print("Tabela 2 — Comparação de Técnicas PEFT para LLaMA-7B")
print("(Adaptado de Hu et al., 2022 e Dettmers et al., 2023)")
print("=" * 80)
print(peft_table.to_string(index=False))

## 9. Tokenização: Impacto no Comprimento de Sequência

Analisamos como o tokenizer do DistilBERT converte textos em tokens,
e qual `max_length` é adequado para nosso dataset.

In [ ]:
# Análise de tokenização
tokenizer = wrapper.tokenizer

token_lengths = []
for text in df["text"].values:
    tokens = tokenizer.encode(text, add_special_tokens=True)
    token_lengths.append(len(tokens))

df["token_count"] = token_lengths

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(token_lengths, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
ax.axvline(x=128, color="red", linestyle="--", label="max_length=128")
ax.set_title("Distribuição de Tokens por Documento (DistilBERT Tokenizer)", fontsize=13)
ax.set_xlabel("Número de Tokens")
ax.set_ylabel("Frequência")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)
print(f"Tokens — média: {np.mean(token_lengths):.0f}, p95: {p95:.0f}, p99: {p99:.0f}, max: {max(token_lengths)}")
print(f"Cobertura com max_length=128: {100*np.mean(np.array(token_lengths) <= 128):.1f}%")

## Resumo

Neste notebook, exploramos:

1. **Dataset**: 5000 textos em 5 domínios (balanceado)
2. **Modelo base**: DistilBERT com ~67M parâmetros
3. **LoRA (r=16)**: Reduz parâmetros treináveis para ~0.2% (Hu et al., 2022)
4. **Trade-off rank**: Maior `r` = mais parâmetros, mais capacidade
5. **Técnicas PEFT**: LoRA e QLoRA oferecem melhor custo-benefício

### Próximo Notebook

No **Notebook 02 (Treinamento)**, implementaremos o fine-tuning completo
com LoRA usando Hugging Face Trainer, comparando com fine-tuning completo.